# ICT-15c — Méta-proxy d'obstruction : structure des désaccords entre proxys

**Issue** : [#7395](https://github.com/jsboige/CoursIA/issues/7395)  
**Epic** : [#4588](https://github.com/jsboige/CoursIA/issues/4588) (IIT -> ICT)

## Cadrage

L'intuition directrice (lecture cohomologie-obstruction, cf. doc Grothendieck
mergée le 19/07/2026) est que **l'obstruction elle-même est informative** :
quand plusieurs proxys de complexité/intégration (spectral, sensitivity,
free-energy) ne « recollent » pas en une mesure globale unique, le *motif*
de leur désaccord porte de l'information sur le substrat.

## Acceptance (issue #7395, falsifiable)

- **Positif** : motif de désaccord **stable cross-substrat** (montrer qu'il
  n'est pas arbitraire — même structure de désaccord sur ≥ 2 substrats).
- **Négatif honnête** : motif = bruit (pas de structure stable) → verdict
  négatif documenté, la jambe meurt proprement.
- **Premier livrable concret** : cas où la lecture-obstruction change une
  décision expérimentale ou une visualisation (pas un habillage rétrospectif).

## Discipline (HARD)

**Ne pas nommer l'objet** tant qu'il n'est pas stabilisé. Le nom
(grothendieckien, « structure » et non « mesure ») vient *après* la
stabilisation empirique, pas avant. Pas de lettre grecque décorée sur un
objet non encore constaté.

## Substrats pilotes (3 instrumentés)

1. **Gray-Scott** (réaction-diffusion, Turing patterns) — `ict.reaction_diffusion.GrayScott`
2. **Axelrod** (morphodynamique stratégique, fin de cycle réplicateur) — `ict.strategic_morphodynamics`
3. **Grokking** (compression crossover) — simulé ici via marche aléatoire biaisée
   sur alphabet à 4 états (proxy minimal).

## Livrable de cette notebook

Application du module `ict.meta_proxy` sur 3 substrats synthétiques,
avec verdict falsifiable `STABLE / NOISE / INCONCLUSIVE` à la fin.

In [1]:
# Imports et substrats pilotes
import sys
from pathlib import Path

ICT_ROOT = Path('.').resolve()
sys.path.insert(0, str(ICT_ROOT))

import numpy as np
import matplotlib
matplotlib.use('Agg')  # rendu PNG en batch
import matplotlib.pyplot as plt

from ict import spectral as SP
from ict import sensitivity as SE
from ict import reaction_diffusion as RD
from ict import strategic_morphodynamics as SM
from ict.meta_proxy import (
    proxy_signature,
    obstruction_vector,
    cross_substrat_obstruction,
)

np.random.seed(20260720)  # determinisme cross-execution
print("meta_proxy loaded. Substrats pilotes : Gray-Scott, Axelrod, Grokking.")

meta_proxy loaded. Substrats pilotes : Gray-Scott, Axelrod, Grokking.


In [2]:
# --- Substrat 1 : Gray-Scott Turing patterns (regime SPOTS non-degenere) ---
# API reelle de ict.reaction_diffusion.GrayScott :
#   - constructeur prend F, k, Du, Dv, dt (PAS de `size`)
#   - seed(n=64, rng=...) -> (U, V) initiaux
#   - run(U, V, steps) -> (U, V, snapshots)
#
# NOTE (fix Prong-B / anti-substrat-degenere) : la premiere version de ce
# notebook (c.752) utilisait F=0.035 / k=0.065 / n=64 / 800 pas avec une
# binarisation a seuil dur V > 0.5. Verifie firsthand : ce regime laisse V
# s'eteindre (V_max = 0.000, V_mean = 0.000, std = 0.000) => ratio = 0.000
# => AUCUN pattern Turing ne se forme. Le seuil V > 0.5 n'est en outre
# JAMAIS atteint en Gray-Scott : l'autocatalyseur V est consomme par la
# reaction, il reste typiquement dans [0.1, 0.4]. Un substrat eteint ne
# peut pas discriminer les proxys : son verdict pollue l'agregation
# cross-substrat (entree degeneree, regle sota-not-workaround Prong-B).
#
# Fix : regime SPOTS (Pearson 1993, F=0.025 / k=0.060), grille n=96,
# 3000 pas, binarisation ADAPTATIVE au seuil V > mean(V) (le pattern se
# definit relativement a l'intensite moyenne, pas a un seuil absolu).
# Verifie firsthand : V_mean = 0.100, V_std = 0.106, ~36% des pixels en
# 'pattern' => substrat NON-degenere, apte a discriminer les proxys.
gs = RD.GrayScott(F=0.025, k=0.060, Du=0.16, Dv=0.08, dt=1.0)
seed_rng = np.random.default_rng(20260720)
U_init, V_init = gs.seed(n=96, rng=seed_rng)
U_final, V_final, _ = gs.run(U_init, V_init, steps=3000)

# Garde-fou anti-degenerescence : on refuse un substrat eteint (std ~ 0).
v = V_final
v_std = float(v.std())
assert v_std > 1e-3, f"Gray-Scott degenere (std={v_std:.4f}) : recalibrer F/k/steps"

# Binarisation adaptative : 'pattern' (1) si V > mean(V), sinon 'background' (0).
# Le seuil absolu V > 0.5 est inappropriate en Gray-Scott (jamais atteint) ;
# la moyenne rend le seuil invariant d'echelle et reflete la structure
# reelle du champ (hauts vs bas relatifs).
v_mean = float(v.mean())
binary_grid = (v > v_mean).astype(int)
n_pattern = int(binary_grid.sum())
n_bg = int((1 - binary_grid).sum())
ratio = n_pattern / (n_pattern + n_bg)
print(f"Gray-Scott SPOTS : V[mean={v_mean:.4f}, std={v_std:.4f}], "
      f"pattern={n_pattern}, background={n_bg}, ratio(>mean)={ratio:.3f}")

# Trajectoire discrete : on scan les pixels ligne par ligne -> sequence de 0/1
gray_scott_states = binary_grid.flatten().tolist()
n_symbols_gs = 2
print(f"Longueur trajectoire Gray-Scott : {len(gray_scott_states)} pixels, "
      f"{len(set(gray_scott_states))} symbole(s) distinct(s)")

Gray-Scott SPOTS : V[mean=0.1000, std=0.1058], pattern=3312, background=5904, ratio(>mean)=0.359
Longueur trajectoire Gray-Scott : 9216 pixels, 2 symbole(s) distinct(s)


In [3]:
# --- Substrat 2 : Axelrod replicator (regime INTRANSITIF non-degenere) ---
# API reelle de ict.strategic_morphodynamics :
#   - make_strategies(rng) -> Dict[str, Strategy]
#   - payoff_matrix(strategies, n_rounds, n_reps, rng) -> A  (matrice de gain)
#   - replicator_trajectory(A, x0, n_steps) -> traj (n_steps+1, n_strategies)
#
# NOTE (fix Prong-B / anti-substrat-degenere) : la premiere version (c.752)
# calibrait A via make_strategies puis x0 uniforme sur 400 pas. Verifie
# firsthand : le replicateur CONVERGE en ~quelques pas puis la dominance
# ne change quasiment plus (counts = {stable: 399, change: 1}). La
# trajectoire binaire est donc quasi-constante => substrat degenere, sa
# signature est ininformative et son verdict pollue l'agregation.
#
# Fix : on utilise un jeu INTRANSITIF pierre-ciseaux-feuille (RPS) dont la
# matrice de gain produit une dominance CYCLIQUE (jamais de convergence ->
# la dynamique reste riche sur toute la trajectoire). Petite asymetrie
# bruitee + point initial perturbe (loin du cycle neutre 1/3) declenchent
# des bascules de dominance reproductibles. Verifie firsthand :
# ~7 bascules / 599 pas => trajectoire NON-degenere.
RPS = np.array([[0.0, 1.0, -1.0],
                [-1.0, 0.0, 1.0],
                [1.0, -1.0, 0.0]])
rng_rps = np.random.default_rng(20260720)
A = RPS + 0.05 * rng_rps.standard_normal((3, 3))
# Le replicateur attend des payoffs positifs : on translate la matrice.
A = A - A.min() + 0.01
n_strat = A.shape[0]
# Point initial perturbe du point fixe symetrique => declenche le cycle.
x0 = np.array([0.45, 0.33, 0.22])
traj = SM.replicator_trajectory(A, x0, n_steps=600)

# Trajectoire discrete : on binarise la stabilite de la dominance.
# Pour chaque pas, on prend l'index de la strategie dominante, puis on
# enregistre 1 si elle change au pas suivant, 0 sinon.
dom_idx = np.argmax(traj, axis=1)
axelrod_states = [int(dom_idx[i] != dom_idx[i - 1]) for i in range(1, len(dom_idx))]
n_changes = int(sum(axelrod_states))
n_symbols_ax = 2

# Garde-fou anti-degenerescence : on refuse une dominance figee.
assert n_changes >= 5, (
    f"Axelrod degenere ({n_changes} bascules/599) : le replicateur a converge ; "
    "augmenter le bruit ou perturber x0 davantage"
)
print(f"Axelrod RPS : {len(axelrod_states)} pas, {n_changes} bascules de "
      f"dominance (regime cyclique non-degenere).")

Axelrod RPS : 600 pas, 7 bascules de dominance (regime cyclique non-degenere).


In [4]:
# --- Substrat 3 : Grokking compression crossover (proxy minimal) ---
# Marche aléatoire biaisée : phase 1 = haute entropie (alphabet 4 etats),
# phase 2 = compression vers un etat attracteur (un seul etat visité).
# Le crossover est placé au pas 200 (milieu de trajectoire).
rng_g = np.random.default_rng(20260720)
n_steps_g = 400
states_g = []
for t in range(n_steps_g):
    if t < 200:
        # Phase haute entropie : equiprobable parmi 4 etats
        s = int(rng_g.integers(0, 4))
    else:
        # Phase compression : 90% du temps sur l'etat 0, 10% sur les autres
        if rng_g.random() < 0.90:
            s = 0
        else:
            s = int(rng_g.integers(1, 4))
    states_g.append(s)
grokking_states = states_g
n_symbols_gk = 4
print(f"Grokking : trajectoire de {len(grokking_states)} pas, crossover au pas 200.")

Grokking : trajectoire de 400 pas, crossover au pas 200.


In [5]:
# --- Calcul des signatures multi-proxys ---
# On enrobe les fonctions ict.spectral / ict.sensitivity pour matcher
# le prototype (states, n_symbols) -> float attendu par meta_proxy.
def spec_gap_wrap(states, n_symbols):
    return float(SP.spectral_summary(states, n_symbols)['spectral_gap'])

def sens_mean_wrap(states, n_symbols):
    # f(x) = x % 2 : fonction bivalente simple
    f = lambda x: x % 2
    return float(SE.sensitivity_distribution(states, n_symbols, f)['mean'])

def sens_max_wrap(states, n_symbols):
    f = lambda x: x % 2
    return float(SE.sensitivity_distribution(states, n_symbols, f)['max'])

sig_gs = proxy_signature(
    gray_scott_states, n_symbols_gs,
    spectral_fn=spec_gap_wrap,
    sensitivity_mean_fn=sens_mean_wrap,
    sensitivity_max_fn=sens_max_wrap,
)
sig_ax = proxy_signature(
    axelrod_states, n_symbols_ax,
    spectral_fn=spec_gap_wrap,
    sensitivity_mean_fn=sens_mean_wrap,
    sensitivity_max_fn=sens_max_wrap,
)
sig_gk = proxy_signature(
    grokking_states, n_symbols_gk,
    spectral_fn=spec_gap_wrap,
    sensitivity_mean_fn=sens_mean_wrap,
    sensitivity_max_fn=sens_max_wrap,
)

print("=== Signatures multi-proxys ===")
for nom, sig in [("gray_scott", sig_gs), ("axelrod", sig_ax), ("grokking", sig_gk)]:
    print(f"{nom:>12}: spectral_gap={sig['spectral_gap']:.4f}, "
          f"sens_mean={sig['sensitivity_mean']:.4f}, "
          f"sens_max={sig['sensitivity_max']:.4f}")

=== Signatures multi-proxys ===
  gray_scott: spectral_gap=0.2107, sens_mean=1.0000, sens_max=1.0000
     axelrod: spectral_gap=1.0118, sens_mean=1.0000, sens_max=1.0000
    grokking: spectral_gap=0.7796, sens_mean=2.0000, sens_max=2.0000


In [6]:
# --- Vecteurs d'obstruction pairwise + aggregation cross-substrat ---
substrats = {"gray_scott": sig_gs, "axelrod": sig_ax, "grokking": sig_gk}
verdict = cross_substrat_obstruction(substrats)

print("=== Vecteurs d'obstruction (a - b normalise) ===")
noms = list(substrats.keys())
for i in range(len(noms)):
    for j in range(i + 1, len(noms)):
        vec = obstruction_vector(substrats[noms[i]], substrats[noms[j]])
        print(f"  {noms[i]:>10} vs {noms[j]:<10}: "
              f"spectral={vec['spectral_gap']:+.3f}, "
              f"sens_mean={vec['sensitivity_mean']:+.3f}, "
              f"sens_max={vec['sensitivity_max']:+.3f}, "
              f"||v||₂={vec['norm_l2']:.3f}")

print(f"\n=== Verdict cross-substrat ===")
print(f"  n_substrats = {verdict['n_substrats']}")
print(f"  pairwise_norms = {[round(x, 4) for x in verdict['pairwise_norms']]}")
print(f"  mean_norm_l2  = {verdict['mean_norm_l2']:.4f}")
print(f"  max_norm_l2   = {verdict['max_norm_l2']:.4f}")
print(f"  STABLE_threshold = {verdict['stable_threshold']}, "
      f"NOISE_threshold = {verdict['noise_threshold']}")
print(f"  VERDICT = {verdict['verdict']}")

=== Vecteurs d'obstruction (a - b normalise) ===
  gray_scott vs axelrod   : spectral=-0.655, sens_mean=+0.000, sens_max=+0.000, ||v||₂=0.655
  gray_scott vs grokking  : spectral=-0.574, sens_mean=-0.333, sens_max=-0.333, ||v||₂=0.743
     axelrod vs grokking  : spectral=+0.130, sens_mean=-0.333, sens_max=-0.333, ||v||₂=0.489

=== Verdict cross-substrat ===
  n_substrats = 3
  pairwise_norms = [0.6553, 0.7431, 0.4889]
  mean_norm_l2  = 0.6291
  max_norm_l2   = 0.7431
  STABLE_threshold = 0.05, NOISE_threshold = 0.3
  VERDICT = NOISE


In [7]:
# --- Visualisation : heatmap des pairwise norms ---
n = len(noms)
M = np.zeros((n, n))
k = 0
for i in range(n):
    for j in range(i + 1, n):
        M[i, j] = verdict['pairwise_norms'][k]
        M[j, i] = M[i, j]
        k += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(M, cmap='viridis', vmin=0, vmax=max(verdict['noise_threshold'], 0.5))
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(noms, rotation=20, ha='right')
ax.set_yticklabels(noms)
for i in range(n):
    for j in range(n):
        if i != j:
            ax.text(j, i, f"{M[i, j]:.3f}", ha='center', va='center',
                    color='white' if M[i, j] < 0.3 else 'black', fontsize=9)
ax.set_title(f"Obstruction pairwise ||v||₂ — verdict : {verdict['verdict']}")
plt.colorbar(im, ax=ax, label='||v||₂ norm')
plt.tight_layout()
plt.savefig('ICT-15c-obstruction-heatmap.png', dpi=110, bbox_inches='tight')
plt.close()
print("Heatmap sauvegardee : ICT-15c-obstruction-heatmap.png")

Heatmap sauvegardee : ICT-15c-obstruction-heatmap.png


## Interprétation

**Substrats testés** (régimes non-dégénérés, fix Prong-B de la v1 c.752) :
Gray-Scott SPOTS (Pearson 1993, F=0.025/k=0.060, binarisation adaptative
`V > mean(V)` — la v1 laissait V s'éteindre, ratio=0.000), Axelrod RPS
(jeu intransitif à dominance cyclique — la v1 convergeait en figeant la
dominance), Grokking (marche aléatoire biaisée, compression crossover à
mi-parcours).

**Proxys calculés** : `spectral_gap` (gap spectral du Laplacien du graphe
de transition), `sensitivity_mean` et `sensitivity_max` (sur fonction
bivalente `f(x) = x % 2`, états visités).

**Vecteur d'obstruction** : `(a - b) / (|a| + |b| + eps)` — invariant à
l'échelle globale, antisymétrique (`obstruction(a, b) = -obstruction(b, a)`).

**Verdict falsifiable** :
- `STABLE` : motif de désaccord cohérent (mean_norm_l₂ ≤ 0.05) sur ≥ 2
  substrats — preuve empirique de l'objet.
- `NOISE` : motif aléatoire (mean_norm_l₂ ≥ 0.30) — verdict négatif honnête,
  la jambe meurt proprement.
- `INCONCLUSIVE` : entre les deux seuils — recalibrer ou augmenter
  l'échantillon.

### Verdict obtenu (sur 3 substrats non-dégénérés)

Le verdict est `NOISE` : les proxys divergent fortement cross-substrat
(mean_norm_l₂ ≈ 0.63, bien au-dessus du seuil NOISE = 0.30). **Ce verdict
est maintenant honnête** — il n'est plus l'artefact d'un Gray-Scott éteint
(v1) ni d'un Axelrod figé (v1), qui produisaient mécaniquement des
signatures triviales. Sur des substrats réellement structurés, les proxys
spectral / sensitivity / max ne se recollent pas en un motif stable : le
désaccord n'est pas porteur d'une structure reproductible.

Ceci valide l'**acceptance négative honnête** de l'issue #7395 : la jambe
« méta-proxy d'obstruction comme objet informatif » ne survit pas à
l'épreuve empirique sur ces trois substrats pilotes. C'est un résultat en
soi (issue #7395, *« Negatif honnete : motif = bruit → verdict negatif
documente, la jambe meurt proprement »*). Une éventuelle relance passerait
par d'autres proxys (free-energy, compression) ou d'autres substrats —
mais l'objet n'a pas, à ce stade, démontré sa stabilisation empirique.

## Suite logique (issue #7395)

Premier livrable concret = au moins un cas où la lecture-obstruction
**change une décision expérimentale ou une visualisation** (issue #7395,
acceptance positive). Sur les 3 substrats pilotes non-dégénérés testés
ici, le verdict est `NOISE` : l'acceptance positive n'est **pas** atteinte,
et l'acceptance négative l'est (documentée honnêtement ci-dessus). La jambe
« méta-proxy obstruction » s'arrête ici à moins d'un changement de jeu de
proxys ou de substrats qui ferait émerger un motif stable.

## Discipline de nommage (HARD)

L'objet n'est **pas nommé** — et ne le sera pas, puisque le verdict `NOISE`
sur substrats non-dégénérés invalide sa stabilisation empirique. Le
vocabulaire reste descriptif (`obstruction vector`, `pairwise norm`,
`STABLE / NOISE`). Aucune lettre grecque décorée sur un objet non constaté.